# M3: Failed Breakout — 完整闭环 Demo

流程:
1. 加载 SPY 数据 → 计算特征
2. 扫描 Failed Breakout 信号
3. 三种退出策略回测
4. 打印指标表
5. 随机抽取信号复盘图

In [1]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')  # headless

from spx_scanner.data_layer import load_data, resample_1m_to_3m
from spx_scanner.features import compute_all_features
from spx_scanner.patterns.registry import get_all_patterns
from spx_scanner.scanner.engine import ScannerEngine
from spx_scanner.backtest.simulator import run_backtest
from spx_scanner.backtest.metrics import compute_metrics
from spx_scanner.viz.chart import plot_signal_context
from spx_scanner.patterns.failed_breakout import FailedBreakout

print('✓ 依赖加载完成')

✓ 依赖加载完成


## 1. 数据 + 特征

In [2]:
%%time
df_1m = load_data('../data/spy_1min.parquet', rth_only=True)
df_3m = resample_1m_to_3m(df_1m)
df = compute_all_features(df_3m)

print(f'数据: {len(df)} bars  |  交易日: {df.index.normalize().nunique()} 天')
print(f'时间范围: {df.index.min().date()} ~ {df.index.max().date()}')

数据: 1820 bars  |  交易日: 14 天
时间范围: 2026-04-06 ~ 2026-04-23
CPU times: user 186 ms, sys: 21.9 ms, total: 208 ms
Wall time: 227 ms


## 2. 扫描信号

In [3]:
pattern = FailedBreakout(symbol='SPY')
engine = ScannerEngine([pattern])
signals_df = engine.scan(df)

print(f'\nFailed Breakout 信号数: {len(signals_df)}')
if not signals_df.empty:
    print(f"  PUT  信号: {(signals_df['direction']=='put').sum()}")
    print(f"  CALL 信号: {(signals_df['direction']=='call').sum()}")
    print(f"  置信度均值: {signals_df['confidence'].mean():.3f}")
    print()
    display_cols = ['timestamp','direction','confidence','entry_price',
                    'stop_level','target_level','ctx_touches']
    display_cols = [c for c in display_cols if c in signals_df.columns]
    print(signals_df[display_cols].to_string(index=False))


Failed Breakout 信号数: 20
  PUT  信号: 19
  CALL 信号: 1
  置信度均值: 0.780

                timestamp direction  confidence  entry_price  stop_level  target_level  ctx_touches
2026-04-07 15:18:00-04:00      call         0.8   657.190002     653.540       664.489            9
2026-04-08 13:21:00-04:00       put         0.8   675.389404     677.482       671.204            7
2026-04-08 13:24:00-04:00       put         0.8   675.219971     677.482       670.695            8
2026-04-08 13:27:00-04:00       put         0.8   675.010010     677.552       669.925            7
2026-04-08 14:18:00-04:00       put         0.9   674.799988     677.788       668.824           16
2026-04-08 14:33:00-04:00       put         0.7   674.864990     677.788       669.019           18
2026-04-08 14:36:00-04:00       put         0.8   674.799988     677.788       668.824           19
2026-04-08 14:42:00-04:00       put         0.8   674.020020     677.823       666.414           20
2026-04-09 14:00:00-04:00       

## 3. 三种退出策略回测

In [4]:
if signals_df.empty:
    print('无信号,跳过回测')
else:
    trades = run_backtest(df, signals_df)
    print(f'交易记录: {len(trades)} 笔 (= {len(signals_df)} 信号 × 3 退出策略)\n')
    
    metrics = compute_metrics(trades)
    print('=== 回测指标 ===')
    pd.set_option('display.float_format', '{:.4f}'.format)
    print(metrics.to_string())
    pd.reset_option('display.float_format')

交易记录: 60 笔 (= 20 信号 × 3 退出策略)

=== 回测指标 ===
                               n_trades  win_rate  avg_win_pct  avg_loss_pct  expect_pct  profit_factor   sharpe  avg_mfe_pct  avg_mae_pct  total_pnl_pct
pattern         exit_strategy                                                                                                                            
failed_breakout fixed_time           20    0.4500       0.1063       -0.1202     -0.0183         0.7240  -3.5220       0.1020      -0.1076        -0.3651
                target_stop          20    0.4500       0.1063       -0.1194     -0.0178         0.7290  -3.4640       0.1020      -0.1076        -0.3559
                trailing_atr         20    0.2500       0.0748       -0.0966     -0.0538         0.2580 -11.4150       0.0822      -0.1037        -1.0756


## 4. 指标可视化:P&L 分布

In [5]:
if not signals_df.empty and not trades.empty:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), facecolor='#1a1a2e')
    strategies = trades['exit_strategy'].unique()
    
    for ax, strategy in zip(axes, strategies):
        grp = trades[trades['exit_strategy'] == strategy]['pnl_pct']
        colors = ['#26a69a' if v >= 0 else '#ef5350' for v in grp]
        ax.bar(range(len(grp)), sorted(grp), color=sorted(colors,
               key=lambda c: (c=='#26a69a')))
        ax.axhline(0, color='white', lw=0.5)
        ax.axhline(grp.mean(), color='yellow', lw=1.2, ls='--',
                   label=f'mean={grp.mean():.3f}%')
        ax.set_facecolor('#1a1a2e')
        ax.tick_params(colors='white')
        ax.set_title(strategy, color='white', fontsize=10)
        ax.set_ylabel('P&L %', color='white')
        ax.legend(facecolor='#222', labelcolor='white', fontsize=8)
        for spine in ax.spines.values(): spine.set_color('#444')
    
    plt.suptitle('Failed Breakout — P&L 分布(每种退出策略)', 
                 color='white', fontsize=12)
    plt.tight_layout()
    plt.savefig('../notebooks/M3_pnl_distribution.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('✓ P&L 分布图已保存')

✓ P&L 分布图已保存


## 5. 信号复盘图(最多 6 个)

In [6]:
if signals_df.empty:
    print('无信号可复盘')
else:
    # 取 fixed_time 策略的交易结果做配对
    ft_trades = trades[trades['exit_strategy'] == 'fixed_time'].copy()
    ft_trades = ft_trades.reset_index(drop=True)
    
    # 获取所有信号的 Signal 对象
    all_signals = pattern.detect(df)
    
    n_review = min(6, len(all_signals))
    rng = np.random.default_rng(42)
    review_idx = rng.choice(len(all_signals), n_review, replace=False)
    review_idx = sorted(review_idx)
    
    for j, sig_idx in enumerate(review_idx):
        sig = all_signals[sig_idx]
        
        # 找对应交易记录
        trade_row = None
        match = ft_trades[ft_trades['entry_time'] == sig.timestamp + pd.Timedelta('3min')]
        if not match.empty:
            trade_row = match.iloc[0].to_dict()
        
        # 打印解释
        print(f'\n--- 信号 {j+1}/{n_review} ---')
        print(pattern.explain(sig))
        if trade_row:
            pnl = trade_row.get('pnl_pct', 0)
            print(f'  回测 P&L: {pnl:+.3f}%  退出原因: {trade_row.get("exit_reason","?")}')  
        
        # 绘图
        fig = plot_signal_context(df, sig, bars_before=25, bars_after=20,
                                   trade_result=trade_row)
        save_path = f'../notebooks/M3_review_{j+1}.png'
        fig.savefig(save_path, dpi=100, bbox_inches='tight')
        plt.close(fig)
        print(f'  图表: {save_path}')
    
    print(f'\n✓ {n_review} 张复盘图已保存')


--- 信号 1/6 ---
[Failed Breakout → PUT]
  阻力位: 676.13  触碰次数: 7
  RVOL: 1.85  BB%分位: 0.131
  EMA 粘合: True  连续下跌: 4 bar
  入场: 675.389  止损: 677.482  目标: 671.204  置信度: 0.80
  回测 P&L: -0.002%  退出原因: time
  图表: ../notebooks/M3_review_1.png

--- 信号 2/6 ---
[Failed Breakout → PUT]
  阻力位: 676.47  触碰次数: 20
  RVOL: 1.86  BB%分位: 0.573
  EMA 粘合: True  连续下跌: 4 bar
  入场: 674.020  止损: 677.823  目标: 666.414  置信度: 0.80
  回测 P&L: +0.089%  退出原因: time
  图表: ../notebooks/M3_review_2.png

--- 信号 3/6 ---
[Failed Breakout → PUT]
  阻力位: 681.016  触碰次数: 10
  RVOL: 1.37  BB%分位: 0.05
  EMA 粘合: True  连续下跌: 1 bar
  入场: 679.720  止损: 682.378  目标: 674.404  置信度: 0.70
  回测 P&L: +0.042%  退出原因: time
  图表: ../notebooks/M3_review_3.png

--- 信号 4/6 ---
[Failed Breakout → PUT]
  阻力位: 681.94  触碰次数: 16
  RVOL: 1.33  BB%分位: 0.02
  EMA 粘合: False  连续下跌: 2 bar
  入场: 681.240  止损: 683.304  目标: 677.112  置信度: 0.80
  回测 P&L: +0.088%  退出原因: time
  图表: ../notebooks/M3_review_4.png

--- 信号 5/6 ---
[Failed Breakout → PUT]
  阻力位: 681.94  触碰次数: 

## 6. M3 验收小结

| 检查项 | 结果 |
|---|---|
| Failed Breakout 信号检测 | ✓ |
| 信号 PUT/CALL 方向正确 | ✓ |
| 止损在阻力位外侧 | ✓ |
| 3 种退出策略回测 | ✓ |
| 指标计算(胜率/期望/MFE/MAE) | ✓ |
| 复盘图(K线+均线+VWAP+入出场标记) | ✓ |
| **pytest** | **99 passed, 0 failed** |

**M3 完成,可进入 M4(扩展 Pattern 库)**